In [1]:
!pip -q install openai datasets pandas scikit-learn tqdm tenacity

In [4]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")
print("API key loaded securely.")

Enter your OpenAI API key: ··········
API key loaded securely.


In [5]:
import os
import json
import time
import pandas as pd

from json import JSONDecodeError
from tqdm import tqdm
from collections import Counter
from datasets import load_dataset
from openai import OpenAI, RateLimitError
from tenacity import retry, wait_exponential, stop_after_attempt, retry_if_exception_type

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

MODEL = "gpt-5.5"
DATASET_NAME = "ChanceFocus/flare-cd"

# First test with 5 or 20.
# Then set LIMIT = None for the full test set.
LIMIT = 100

dataset = load_dataset(DATASET_NAME)

split_name = "test" if "test" in dataset else list(dataset.keys())[0]
split_data = dataset[split_name]

print(dataset)
print("Split:", split_name)
print("Columns:", split_data.column_names)
print("Example:", split_data[0])

VALID_LABELS = [
    "O",
    "B-CAUSE", "I-CAUSE",
    "B-EFFECT", "I-EFFECT"
]

LABEL_SCHEMA = {
    "type": "object",
    "properties": {
        "labels": {
            "type": "array",
            "items": {
                "type": "string",
                "enum": VALID_LABELS
            }
        }
    },
    "required": ["labels"],
    "additionalProperties": False
}

SYSTEM_PROMPT = """
You are a cause-effect sequence labeling model.

Task:
Given a list of tokens from a text section, assign one BIO label to each token.

Allowed labels:
- O
- B-CAUSE, I-CAUSE
- B-EFFECT, I-EFFECT

Definitions:
- CAUSE: the text span that represents the cause of an event.
- EFFECT: the text span that represents the effect or result caused by the cause.
- O: tokens that are not part of a cause or effect span.

Rules:
1. Return exactly one label for each input token.
2. The number of output labels must equal the number of input tokens.
3. Use BIO format.
4. Do not add explanations.
5. Return only valid JSON following the schema.
"""

def get_column_name(example, possible_names):
    for name in possible_names:
        if name in example:
            return name

    raise ValueError(
        f"Cannot find any column from {possible_names}. "
        f"Available columns are: {list(example.keys())}"
    )

def normalize_label(label):
    label = str(label).strip()

    if label in VALID_LABELS:
        return label

    return "O"

def fix_prediction_length(pred_labels, target_len):
    pred_labels = [normalize_label(x) for x in pred_labels]

    if len(pred_labels) < target_len:
        pred_labels = pred_labels + ["O"] * (target_len - len(pred_labels))

    if len(pred_labels) > target_len:
        pred_labels = pred_labels[:target_len]

    return pred_labels

def bio_to_spans(labels):
    spans = []
    start = None
    span_type = None

    for i, label in enumerate(labels + ["O"]):
        if label == "O":
            if start is not None:
                spans.append((start, i, span_type))
                start = None
                span_type = None
            continue

        if "-" not in label:
            continue

        prefix, label_type = label.split("-", 1)

        if prefix == "B":
            if start is not None:
                spans.append((start, i, span_type))
            start = i
            span_type = label_type

        elif prefix == "I":
            if start is None:
                start = i
                span_type = label_type
            elif span_type != label_type:
                spans.append((start, i, span_type))
                start = i
                span_type = label_type

    return spans

def compute_metrics(gold_all, pred_all):
    correct_tokens = 0
    total_tokens = 0
    exact_match_count = 0

    tp = 0
    fp = 0
    fn = 0

    for gold_labels, pred_labels in zip(gold_all, pred_all):
        total_tokens += len(gold_labels)
        correct_tokens += sum(g == p for g, p in zip(gold_labels, pred_labels))

        if gold_labels == pred_labels:
            exact_match_count += 1

        gold_spans = Counter(bio_to_spans(gold_labels))
        pred_spans = Counter(bio_to_spans(pred_labels))

        tp += sum((gold_spans & pred_spans).values())
        fp += sum((pred_spans - gold_spans).values())
        fn += sum((gold_spans - pred_spans).values())

    precision = tp / (tp + fp) if tp + fp > 0 else 0.0
    recall = tp / (tp + fn) if tp + fn > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0.0
    token_accuracy = correct_tokens / total_tokens if total_tokens > 0 else 0.0
    exact_match_accuracy = exact_match_count / len(gold_all) if len(gold_all) > 0 else 0.0

    return {
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
        "token_accuracy": token_accuracy,
        "exact_match_accuracy": exact_match_accuracy,
        "true_positive": tp,
        "false_positive": fp,
        "false_negative": fn
    }

@retry(
    retry=retry_if_exception_type((RateLimitError, JSONDecodeError, ValueError)),
    wait=wait_exponential(multiplier=5, min=10, max=120),
    stop=stop_after_attempt(5)
)
def call_openai_cd(tokens):
    indexed_tokens = "\n".join([f"{i}: {tok}" for i, tok in enumerate(tokens)])

    response = client.responses.create(
        model=MODEL,
        input=[
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": (
                    f"Number of tokens: {len(tokens)}\n\n"
                    f"Tokens:\n{indexed_tokens}\n\n"
                    f"Return exactly {len(tokens)} BIO labels."
                )
            }
        ],
        text={
            "format": {
                "type": "json_schema",
                "name": "flare_cd_sequence_labeling",
                "schema": LABEL_SCHEMA,
                "strict": True
            }
        },
        reasoning={
            "effort": "low"
        },
        max_output_tokens=1200
    )

    raw_output = response.output_text

    if raw_output is None or raw_output.strip() == "":
        raise ValueError("Empty model output")

    try:
        parsed = json.loads(raw_output)
    except JSONDecodeError:
        print("Raw model output:")
        print(repr(raw_output))
        raise

    if "labels" not in parsed:
        raise ValueError(f"Missing labels field in model output: {parsed}")

    return parsed

example = split_data[0]

token_col = get_column_name(example, ["token", "tokens"])
label_col = get_column_name(example, ["label", "labels", "tags"])

print("Token column:", token_col)
print("Label column:", label_col)

eval_data = split_data if LIMIT is None else split_data.select(range(LIMIT))

rows = []
gold_all = []
pred_all = []

for idx, row in enumerate(tqdm(eval_data)):
    tokens = row[token_col]
    gold_labels = [normalize_label(x) for x in row[label_col]]

    try:
        result = call_openai_cd(tokens)
        pred_labels = result.get("labels", [])
        pred_labels = fix_prediction_length(pred_labels, len(gold_labels))
        error = ""
    except Exception as e:
        pred_labels = ["O"] * len(gold_labels)
        error = str(e)
        print(f"Error at row {idx}: {e}")

    gold_all.append(gold_labels)
    pred_all.append(pred_labels)

    rows.append({
        "idx": idx,
        "id": row.get("id", ""),
        "text": row.get("text", ""),
        "tokens": tokens,
        "gold_labels": gold_labels,
        "pred_labels": pred_labels,
        "gold_spans": bio_to_spans(gold_labels),
        "pred_spans": bio_to_spans(pred_labels),
        "exact_match": gold_labels == pred_labels,
        "error": error
    })

    # Slow down to reduce RateLimitError.
    time.sleep(5)

metrics = compute_metrics(gold_all, pred_all)

print("\n===== GPT-5.5 FLARE-CD Evaluation =====")
print(f"Dataset: {DATASET_NAME}")
print(f"Split: {split_name}")
print(f"Model: {MODEL}")
print(f"Samples evaluated: {len(eval_data)}")
print(f"Precision / Correctness Rate: {metrics['precision']:.4f}")
print(f"Recall: {metrics['recall']:.4f}")
print(f"F1 Score: {metrics['f1_score']:.4f}")
print(f"Exact Match Accuracy: {metrics['exact_match_accuracy']:.4f}")
print(f"Token Accuracy: {metrics['token_accuracy']:.4f}")
print(f"TP: {metrics['true_positive']}")
print(f"FP: {metrics['false_positive']}")
print(f"FN: {metrics['false_negative']}")

df = pd.DataFrame(rows)
df.to_csv("gpt55_flare_cd_evaluation_results.csv", index=False)

with open("gpt55_flare_cd_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

num_errors = df["error"].astype(str).str.len().gt(0).sum()

print("\n===== Error Summary =====")
print(f"Failed samples: {num_errors}")
print(f"Total samples: {len(df)}")
print(f"Failure rate: {num_errors / len(df):.4f}")

pd.set_option("display.max_colwidth", 120)
df.head()

DatasetDict({
    test: Dataset({
        features: ['id', 'query', 'answer', 'text', 'label', 'token'],
        num_rows: 226
    })
})
Split: test
Columns: ['id', 'query', 'answer', 'text', 'label', 'token']
Example: {'id': 'cd0', 'query': "Your job in this task is to perform sequence labeling on a provided text section, marking the chunks that represent the cause of an event and the effects that result from it. For each token in the text, assign a label to indicate its role in representing cause or effect. The labels you should use are 'B-CAUSE', 'I-CAUSE', 'B-EFFECT', 'I-EFFECT', and 'O'. A 'B-' prefix is used to denote the beginning of a cause or effect sequence, while an 'I-' prefix is used for continuation of a cause or effect sequence. If a token is not part of either a cause or effect sequence, label it as 'O'. Provide your answer as a sequence of 'token:label' pairs, with each pair on a new line.\nText: Around 21,000 employees , 9,000 of whom are employed in the UK , are to b

100%|██████████| 100/100 [17:30<00:00, 10.50s/it]


===== GPT-5.5 FLARE-CD Evaluation =====
Dataset: ChanceFocus/flare-cd
Split: test
Model: gpt-5.5
Samples evaluated: 100
Precision / Correctness Rate: 0.2358
Recall: 0.2500
F1 Score: 0.2427
Exact Match Accuracy: 0.0600
Token Accuracy: 0.4426
TP: 50
FP: 162
FN: 150

===== Error Summary =====
Failed samples: 0
Total samples: 100
Failure rate: 0.0000


,idx,id,text,tokens,gold_labels,pred_labels,gold_spans,pred_spans,exact_match,error
0,0,cd0,"Around 21,000 employees , 9,000 of whom are employed in the UK , are to be made redundant after the 178-year-old com...","[Around, 21,000, employees, ,, 9,000, of, whom, are, employed, in, the, UK, ,, are, to, be, made, redundant, after, ...","[B-EFFECT, I-EFFECT, I-EFFECT, I-EFFECT, I-EFFECT, I-EFFECT, I-EFFECT, I-EFFECT, I-EFFECT, I-EFFECT, I-EFFECT, I-EFF...","[B-EFFECT, I-EFFECT, I-EFFECT, I-EFFECT, I-EFFECT, I-EFFECT, I-EFFECT, I-EFFECT, I-EFFECT, I-EFFECT, I-EFFECT, I-EFF...","[(0, 18, EFFECT), (19, 31, CAUSE)]","[(0, 18, EFFECT), (19, 31, CAUSE)]",True,
1,1,cd1,REUTERS/Aly Song/File Photo ( Reuters ) - Tencent Holdings Ltd and private equity partner Hammer Capital have offere...,"[REUTERS/Aly, Song/File, Photo, (, Reuters, ), -, Tencent, Holdings, Ltd, and, private, equity, partner, Hammer, Cap...","[O, O, O, O, O, O, O, B-CAUSE, I-CAUSE, I-CAUSE, I-CAUSE, I-CAUSE, I-CAUSE, I-CAUSE, I-CAUSE, I-CAUSE, I-CAUSE, I-CA...","[O, O, O, O, O, O, O, B-CAUSE, I-CAUSE, I-CAUSE, I-CAUSE, I-CAUSE, I-CAUSE, I-CAUSE, I-CAUSE, I-CAUSE, I-CAUSE, I-CA...","[(7, 36, CAUSE), (37, 46, EFFECT)]","[(7, 36, CAUSE), (37, 46, EFFECT)]",True,
2,2,cd2,"Finally , Bank of America reduced their price target on Intrexon from $ 7.00 to $ 6.00 and set an underperform ratin...","[Finally, ,, Bank, of, America, reduced, their, price, target, on, Intrexon, from, $, 7.00, to, $, 6.00, and, set, a...","[B-CAUSE, I-CAUSE, I-CAUSE, I-CAUSE, I-CAUSE, I-CAUSE, I-CAUSE, I-CAUSE, I-CAUSE, I-CAUSE, I-CAUSE, I-CAUSE, I-CAUSE...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, B-CAUSE, I-CA...","[(0, 34, CAUSE), (64, 81, EFFECT)]","[(34, 63, CAUSE), (64, 81, EFFECT)]",False,
3,3,cd3,"RWR traded up $ 0.29 during trading on Wednesday , hitting $ 103.68.","[RWR, traded, up, $, 0.29, during, trading, on, Wednesday, ,, hitting, $, 103.68.]","[B-CAUSE, I-CAUSE, I-CAUSE, I-CAUSE, I-CAUSE, I-CAUSE, I-CAUSE, I-CAUSE, I-CAUSE, O, B-EFFECT, I-EFFECT, I-EFFECT]","[B-CAUSE, I-CAUSE, I-CAUSE, I-CAUSE, I-CAUSE, I-CAUSE, I-CAUSE, I-CAUSE, I-CAUSE, O, B-EFFECT, I-EFFECT, I-EFFECT]","[(0, 9, CAUSE), (10, 13, EFFECT)]","[(0, 9, CAUSE), (10, 13, EFFECT)]",True,
4,4,cd4,"More than 20,000 jobs across the group are at risk if it collapses , with 9,000 of those jobs in the UK.","[More, than, 20,000, jobs, across, the, group, are, at, risk, if, it, collapses, ,, with, 9,000, of, those, jobs, in...","[B-EFFECT, I-EFFECT, I-EFFECT, I-EFFECT, I-EFFECT, I-EFFECT, I-EFFECT, I-EFFECT, I-EFFECT, I-EFFECT, O, B-CAUSE, I-C...","[B-EFFECT, I-EFFECT, I-EFFECT, I-EFFECT, I-EFFECT, I-EFFECT, I-EFFECT, I-EFFECT, I-EFFECT, I-EFFECT, O, B-CAUSE, I-C...","[(0, 10, EFFECT), (11, 22, CAUSE)]","[(0, 10, EFFECT), (11, 13, CAUSE), (15, 22, EFFECT)]",False,
